# Voronoi Diagrams and 1-Nearest Neighbour Classification

**Audience:** students who have just seen the supervised-learning setup.

**What you will build:** an interactive exploration of the Voronoi diagram that
underlies the 1-nearest-neighbour classifier, with NumPy, matplotlib and SciPy.

## Learning objectives

By the end of this notebook you will be able to

1. define a Voronoi diagram and explain how it partitions the input space;
2. see that the 1-NN classifier's decision boundary is exactly the set of Voronoi
   edges between differently labelled cells;
3. generate Voronoi diagrams from arbitrary seed points;
4. relate the geometry of the cells to the classifier's bias--variance trade-off.

## Why it matters

The lecture introduces the nearest-neighbour rule as "find the closest training
point and output its label." That rule carves the input space into regions, one
around every training point, such that every location in a given region is closer
to its associated training point than to any other. That partition is called a
**Voronoi diagram**. Understanding it makes the 1-NN classifier completely
transparent: there is nothing hidden happening.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi, voronoi_plot_2d
from sklearn.neighbors import KNeighborsClassifier

np.random.seed(42)
plt.rcParams.update(
    {
        "figure.figsize": (8, 6),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Setup complete")

## 1. Construct a Voronoi diagram from arbitrary seeds

`scipy.spatial.Voronoi` takes an array of seed points (the training inputs) and
returns the finite `vertices` and the `ridge_vertices` that connect them. A ridge
with an index of `-1` extends to infinity. `voronoi_plot_2d` draws the whole
diagram for us; we then reproduce the same geometry by hand in the next section.

In [ ]:
rng = np.random.default_rng(0)
seeds = rng.uniform(0.0, 1.0, size=(12, 2))
labels = rng.integers(0, 2, size=len(seeds))

vor = Voronoi(seeds)
fig, ax = plt.subplots()
voronoi_plot_2d(vor, ax=ax, show_vertices=False, line_colors="0.4",
                line_width=1.2, point_size=6)
ax.scatter(seeds[:, 0], seeds[:, 1], c=labels, cmap="coolwarm",
           edgecolors="k", s=70, zorder=3)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x1", ylabel="x2",
       title="Voronoi diagram of 12 seed points")
plt.show()

print(f"finite vertices: {len(vor.vertices)}, ridges: {len(vor.ridge_points)}")

## 2. 1-NN decision regions are Voronoi cells

Classify every point on a fine grid by its nearest seed. The resulting STEP
function is constant inside each Voronoi cell and jumps exactly on the ridges.
This is the decision surface of a 1-nearest-neighbour classifier.

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(0, 1, 400), np.linspace(0, 1, 400))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])

knn = KNeighborsClassifier(n_neighbors=1).fit(seeds, labels)
decisions = knn.predict(grid).reshape(grid_x.shape)

fig, ax = plt.subplots()
ax.imshow(decisions, origin="lower", extent=(0, 1, 0, 1), cmap="coolwarm",
          alpha=0.35, interpolation="nearest")
voronoi_plot_2d(vor, ax=ax, show_vertices=False, show_points=False,
                line_colors="k", line_width=0.6)
ax.scatter(seeds[:, 0], seeds[:, 1], c=labels, cmap="coolwarm",
           edgecolors="k", s=70, zorder=3)
ax.set(xlim=(0, 1), ylim=(0, 1), title="1-NN regions = Voronoi cells")
plt.show()

## 3. Only edges between different labels are decision boundaries

A ridge connecting two seeds with the **same** label is invisible to the
classifier: crossing it does not change the predicted class. We can filter the
ridge list to keep only differently labelled pairs and overlay the result on the
boundary obtained by thresholding the 1-NN predictions.

In [ ]:
def boundary_segments(voronoi, seed_labels):
    """Return line segments of ridges separating differently labelled seeds."""
    segments = []
    for (i, j), (a, b) in zip(voronoi.ridge_points, voronoi.ridge_vertices):
        if seed_labels[i] == seed_labels[j] or a < 0 or b < 0:
            continue
        segments.append(voronoi.vertices[[a, b]])
    return segments


segments = boundary_segments(vor, labels)
fig, ax = plt.subplots()
ax.contour(grid_x, grid_y, decisions, levels=[0.5], colors="tab:blue",
           linewidths=1)
for segment in segments:
    ax.plot(segment[:, 0], segment[:, 1], color="red", linestyle="--",
            linewidth=2, label="Voronoi ridge (different labels)")
ax.scatter(seeds[:, 0], seeds[:, 1], c=labels, cmap="coolwarm",
           edgecolors="k", s=70, zorder=3)
handles, legend_labels = ax.get_legend_handles_labels()
if handles:
    ax.legend(handles[:1], legend_labels[:1], loc="upper right", fontsize=8)
ax.set(title="Blue: 1-NN threshold; red: Voronoi boundary edges",
       xlabel="x1", ylabel="x2")
plt.show()

print(f"{len(segments)} separating ridges out of {len(vor.ridge_points)} total")

## 4. Cell geometry and the bias--variance trade-off

With few seeds the cells are large and the classifier has high bias: it can only
represent a coarse partition. With many seeds the cells become small and
contorted, so a single noisy label can carve out an isolated island: variance is
high. The experiment below samples training sets of increasing size and measures
the 1-NN error on a fixed test grid.

In [ ]:
def target(x):
    return ((x[:, 0] > 0.5) ^ (x[:, 1] > 0.5)).astype(int)


test_points = rng.uniform(0, 1, size=(600, 2))
test_labels = target(test_points)
sizes = [2, 5, 10, 20, 50, 100, 200]
errors = []
for n in sizes:
    train_points = rng.uniform(0, 1, size=(n, 2))
    model = KNeighborsClassifier(n_neighbors=1).fit(
        train_points, target(train_points)
    )
    errors.append(np.mean(model.predict(test_points) != test_labels))

fig, ax = plt.subplots()
ax.semilogx(sizes, errors, marker="o")
ax.set(xlabel="Number of seeds (training points)", ylabel="Test error",
       title="1-NN learning curve on a checkerboard target")
plt.show()
print(dict(zip(sizes, np.round(errors, 3))))

## Exercises

1. Replace the random seeds with points on a regular grid. How does the Voronoi
   diagram change, and what does that imply about the classifier's bias?
2. Add a duplicate seed point. Why does its Voronoi cell vanish?
3. Increase `k` in the classifier. Verify that the boundary is no longer made of
   single Voronoi edges but of order-`k` Voronoi regions.
4. Corrupt 20% of the seed labels. Re-run the learning curve and explain the
   increase in variance.

The `.dat` files in `Lecture Notes/Files/Diagrams/` contain a small Voronoi grid,
points and boundary that are used in the lecture slides.